# Analisis Sentimen Pengguna Aplikasi Gojek Berdasarkan Ulasan Google Play Store pada Periode 1–25 Mei 2026

# **IMPORT DATASET**

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('scraping data.xlsx') #membaca data set

In [ ]:
df.head() # menampilkan 5 data teratas

,reviewId,userName,score,content,at,thumbsUpCount,reviewCreatedVersion,replyContent,repliedAt
0,33119429-6fb7-4ed1-b27c-0aaf27ad62ca,Ilham Koko,1,driver lu pada gak niat kerja 2 jam gua nunggu...,2026-05-24 20:47:25,0,5411.0,NaN,NaN
1,ad204955-701c-46fb-900e-05c32c7eb8ca,Ahmat Sadikin,1,memang tarifnya murah.. tapi mencari drivernya...,2026-05-24 20:46:59,0,5611.0,NaN,NaN
2,144d365b-998b-4bc1-9281-3588d6603e56,Fazar Nasrullah,1,drivernya gamau ambil orderan yang hemat,2026-05-24 20:34:30,0,5601.0,NaN,NaN
3,b89b72ce-2088-4caf-819d-1c537bfbdca7,Ny. Juwarni,5,sangat membantu cepat dan profesional,2026-05-24 20:23:48,0,NaN,NaN,NaN
4,470705db-f3f1-4f28-b9b4-43af8b20b599,adi Maskuri,1,"saya korban penipuan dari fitur gojek pinjam, ...",2026-05-24 20:17:59,0,NaN,NaN,NaN


In [ ]:
print("===info dataset ====")
print(f"Jumlah baris : {len(df):,} ")
print(f"Jumlah kolom : {len(df.columns)} ")
print(f"kolom        : {df.columns.tolist()}")
print()

===info dataset ====
Jumlah baris : 3,007 
Jumlah kolom : 9 
kolom        : ['reviewId', 'userName', 'score', 'content', 'at', 'thumbsUpCount', 'reviewCreatedVersion', 'replyContent', 'repliedAt']



In [ ]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())
print()


=== MISSING VALUES ===
reviewId                   0
userName                   0
score                      0
content                    0
at                         0
thumbsUpCount              0
reviewCreatedVersion     705
replyContent            3007
repliedAt               3007
dtype: int64



In [ ]:
print("=== DISTRIBUSI RATING ===")
print(df['score'].value_counts().sort_index())
print()

=== DISTRIBUSI RATING ===
score
1     621
2     110
3      99
4     172
5    2005
Name: count, dtype: int64



analisis

PREPROCESSING TEKS

In [ ]:
def labl_sentimen(score):
  if score >=4:
    return 'Positif'
  elif score <=2:
    return 'Negatif'
  else:
    return 'Netral'

df['sentiment'] = df['score'].apply(labl_sentimen)
print(df['sentiment'].value_counts())
print()

sentiment
Positif    2177
Negatif     731
Netral       99
Name: count, dtype: int64



# ANALISIS DARI RATING :

Pengguna memberikan sentimen yang positif terhadap Aplikasi Gojek dari tanggal 1 - 05 - 2026 hingga 25 - 05 - 2026.



*   72,4 % (2177)ulasan Positif
*   24,3 % (731)ulasan bersifat Negatif
*   3,3 % (99)ulasan bersifat Netral

dari sentimen ini menunjukkan bahwa sebagian besar para pengguna merasa puas terhadap pelayanan produk yang diberikan. Tingginya jumlah ulasan positif menandakan bahwa pengalaman sebagian pengguna baik.

namun, masih adanya beberapa ulasan negatif yang cukup signifikan, angka dari ulasan negatif tersebut yaitu 700 lebih. ini memnandakan bahwa beberapa pengguna masih mengeluhkan pelayanan dari aplikasi gojek. seperti perlayanan, performa kurang baik, bug pada produk, dan lain - lain.

lalu pada ulasan netral memiliki jumlah yang paling sedikit, yang artinya setiap pengguna cenderung memberikan ulasan yang jelas, baik puas maupun tidak puas.


In [ ]:
#menghapus kata kata pada ulasan yang tidak memiliki makna penting dalam menganalisis.
STOPWORDS = {
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'tidak',
    'dengan', 'saya', 'aja', 'gak', 'ga', 'ya', 'aku', 'bisa',
    'juga', 'sih', 'kalo', 'tapi', 'udah', 'ada', 'banget',
    'kalau', 'nih', 'buat', 'lagi', 'mau', 'sama', 'karena',
    'tp', 'jd', 'dh', 'lg', 'ngga', 'pun', 'kok', 'tdk', 'tau',
    'deh', 'mah', 'loh', 'pas', 'klo', 'emang', 'pake', 'sampe',
    'lebih', 'jadi', 'krn', 'pd', 'gitu', 'kan', 'si', 'lah',
    'harus', 'makin', 'karna'
}

In [ ]:
def preprocess(text):
    text = str(text).lower()                        # lowercase
    text = re.sub(r'[^a-z\s]', '', text)            # hapus angka & tanda baca
    tokens = text.split()                            # tokenisasi
    tokens = [w for w in tokens                      # hapus stopwords
              if w not in STOPWORDS and len(w) > 2]
    return ' '.join(tokens)

df['content_clean'] = df['content'].apply(preprocess)


print("=== CONTOH PREPROCESSING ===")
for _, row in df[df['score']==1].head(2).iterrows():
    print(f"ASLI   : {row['content'][:80]}...")
    print(f"BERSIH : {row['content_clean'][:80]}...")
    print()


def top_words(subset, n=10):
    all_text = ' '.join(subset['content_clean'])
    freq = Counter(all_text.split())
    return freq.most_common(n)

print("=== TOP KATA — REVIEW NEGATIF ===")
print(top_words(df[df['sentiment']=='Negatif']))

print("\n=== TOP KATA — REVIEW POSITIF ===")
print(top_words(df[df['sentiment']=='Positif']))

=== CONTOH PREPROCESSING ===
ASLI   : driver lu pada gak niat kerja 2 jam gua nunggu driver, ongkir lu mahal gak jelas...
BERSIH : driver pada niat kerja jam gua nunggu driver ongkir mahal jelas...

ASLI   : memang tarifnya murah.. tapi mencari drivernya lama, bisa sampai 10 - 30 menit ....
BERSIH : memang tarifnya murah mencari drivernya lama sampai menit jika sudah memilih men...

=== TOP KATA — REVIEW NEGATIF ===
[('driver', 440), ('nya', 205), ('gojek', 200), ('aplikasi', 192), ('lama', 126), ('nunggu', 95), ('padahal', 91), ('malah', 90), ('jam', 85), ('gofood', 83)]

=== TOP KATA — REVIEW POSITIF ===
[('sangat', 289), ('bagus', 254), ('gojek', 211), ('membantu', 203), ('mantap', 190), ('driver', 135), ('aplikasi', 121), ('nya', 116), ('cepat', 108), ('baik', 104)]


In [ ]:
df_ml = df[df['sentiment'] != 'Netral'].copy()
df_ml['label'] = (df_ml['sentiment'] == 'Positif').astype(int)
# label: 1 = Positif, 0 = Negatif

print(f"\n=== DATA ML ===")
print(f"Total data   : {len(df_ml):,}")
print(f"Negatif (0)  : {(df_ml['label']==0).sum():,}")
print(f"Positif (1)  : {(df_ml['label']==1).sum():,}")



=== DATA ML ===
Total data   : 2,908
Negatif (0)  : 731
Positif (1)  : 2,177


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_ml['content_clean'],
    df_ml['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_ml['label']   # menjaga proporsi kelas
)
print(f"\nTraining : {len(X_train):,} data")
print(f"Testing  : {len(X_test):,} data")


Training : 2,326 data
Testing  : 582 data


In [ ]:
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f"\nShape matrix TF-IDF training : {X_train_vec.shape}")


Shape matrix TF-IDF training : (2326, 3000)


In [ ]:

model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
model.fit(X_train_vec, y_train)


y_pred = model.predict(X_test_vec)

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"\n=== HASIL EVALUASI ===")
print(f"Akurasi Model : {acc:.2%}")
print()


=== HASIL EVALUASI ===
Akurasi Model : 88.83%



In [ ]:
print(classification_report(y_test, y_pred,
      target_names=['Negatif', 'Positif']))

              precision    recall  f1-score   support

     Negatif       0.81      0.72      0.76       146
     Positif       0.91      0.94      0.93       436

    accuracy                           0.89       582
   macro avg       0.86      0.83      0.85       582
weighted avg       0.89      0.89      0.89       582



In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Negatif','Positif'],
            yticklabels=['Negatif','Positif'])
plt.title(f'Confusion Matrix (Akurasi: {acc:.2%})')
plt.xlabel('Prediksi'); plt.ylabel('Aktual')
plt.tight_layout()
plt.savefig('plot_2_confusion_matrix.png', dpi=120)
plt.close()

In [ ]:
feat_names = tfidf.get_feature_names_out()
coefs = model.coef_[0]
top_pos = [(feat_names[i], coefs[i])
           for i in coefs.argsort()[-10:][::-1]]
top_neg = [(feat_names[i], abs(coefs[i]))
           for i in coefs.argsort()[:10]]

print("=== KATA PALING BERPENGARUH ===")
print("→ Mengarah ke POSITIF:")
for w, c in top_pos:
    print(f"   {w:<20} koef: {c:.3f}")
print("→ Mengarah ke NEGATIF:")
for w, c in top_neg:
    print(f"   {w:<20} koef: {c:.3f}")

=== KATA PALING BERPENGARUH ===
→ Mengarah ke POSITIF:
   bagus                koef: 2.969
   mantap               koef: 2.725
   membantu             koef: 2.342
   good                 koef: 1.910
   sangat               koef: 1.810
   baik                 koef: 1.800
   oke                  koef: 1.797
   cepat                koef: 1.758
   ramah                koef: 1.712
   keren                koef: 1.665
→ Mengarah ke NEGATIF:
   driver               koef: 3.878
   aplikasi             koef: 2.579
   malah                koef: 2.429
   padahal              koef: 2.403
   buruk                koef: 2.377
   jelek                koef: 1.984
   gofood               koef: 1.953
   jam                  koef: 1.867
   jelas                koef: 1.841
   susah                koef: 1.811


# Analisis Kata Paling Berpengaruh

Berdasarkan hasil analisis, kata yang paling berpengaruh terhadap sentimen positif adalah “bagus”, “mantap”, “membantu”, dan “cepat”. Hal ini menunjukkan bahwa pengguna merasa puas terhadap kualitas layanan dan kemudahan penggunaan aplikasi Gojek.

Sedangkan pada sentimen negatif, kata seperti “driver”, “aplikasi”, “buruk”, dan “susah” paling sering muncul. Hal ini mengindikasikan bahwa keluhan pengguna banyak berkaitan dengan pelayanan driver serta performa aplikasi.

In [ ]:
def prediksi_sentimen(teks):
    bersih  = preprocess(teks)
    vektor  = tfidf.transform([bersih])
    pred    = model.predict(vektor)[0]
    proba   = model.predict_proba(vektor)[0]
    label   = 'Positif ✅' if pred == 1 else 'Negatif ❌'
    conf    = max(proba)
    return f"Sentimen: {label} (Keyakinan: {conf:.1%})"

print("\n=== DEMO PREDIKSI ===")
contoh = [
    "aplikasinya sangat bagus dan mudah digunakan, driver cepat datang",
    "driver cancel terus, nunggu 1 jam ga dapat, sangat mengecewakan",
    "lumayan membantu tapi harga makin mahal",
]
for c in contoh:
    print(f"Teks : {c[:60]}...")
    print(f"       {prediksi_sentimen(c)}\n")


=== DEMO PREDIKSI ===
Teks : aplikasinya sangat bagus dan mudah digunakan, driver cepat d...
       Sentimen: Positif ✅ (Keyakinan: 91.5%)

Teks : driver cancel terus, nunggu 1 jam ga dapat, sangat mengecewa...
       Sentimen: Negatif ❌ (Keyakinan: 64.4%)

Teks : lumayan membantu tapi harga makin mahal...
       Sentimen: Positif ✅ (Keyakinan: 82.9%)

